# 🎨 Data Designer Tutorial: Seeding Synthetic Data Generation with an External Dataset

#### 📚 What you'll learn

In this notebook, we will demonstrate how to seed synthetic data generation in Data Designer with an external dataset.

If this is your first time using Data Designer, we recommend starting with the [first notebook](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/1-the-basics/) in this tutorial series.


# 🎨 Data Designer チュートリアル：外部データセットを使用した合成データ生成のシード設定

#### 📚 学習内容

このノートブックでは、Data Designer で外部データセットを使用して合成データ生成のシードを設定する方法を説明します。

Data Designer を初めて使用する場合は、このチュートリアルシリーズの [最初のノートブック](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/1-the-basics/) から始めることをお勧めします。


### 📦 Import Data Designer

- `data_designer.config` provides access to the configuration API.

- `DataDesigner` is the main interface for data generation.


### 📦 データデザイナーのインポート

- `data_designer.config` は設定APIへのアクセスを提供します。

- `DataDesigner` はデータ生成のためのメインインターフェースです。


In [1]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### ⚙️ Initialize the Data Designer interface

- `DataDesigner` is the main object responsible for managing the data generation process.

- When initialized without arguments, the [default model providers](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) are used.


### ⚙️ データデザイナーインターフェースの初期化

- `DataDesigner` は、データ生成プロセスを管理する主要なオブジェクトです。

- 引数を指定せずに初期化した場合、[デフォルトのモデルプロバイダー](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) が使用されます。


In [2]:
data_designer = DataDesigner()

### 🎛️ Define model configurations

- Each `ModelConfig` defines a model that can be used during the generation process.

- The "model alias" is used to reference the model in the Data Designer config (as we will see below).

- The "model provider" is the external service that hosts the model (see the [model config](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) docs for more details).

- By default, we use [build.nvidia.com](https://build.nvidia.com/models) as the model provider.



### 🎛️ モデル設定の定義

- 各 `ModelConfig` は、生成プロセスで使用できるモデルを定義します。

- 「モデルエイリアス」は、Data Designer の設定でモデルを参照するために使用されます（後述します）。

- 「モデルプロバイダ」は、モデルをホストする外部サービスです（詳細は [モデル設定](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) のドキュメントを参照してください）。

- デフォルトでは、モデルプロバイダとして [build.nvidia.com](https://build.nvidia.com/models) を使用します。


In [3]:
# This name is set in the model provider configuration.
MODEL_PROVIDER = "nvidia"

# The model ID is from build.nvidia.com.
MODEL_ID = "nvidia/nemotron-3-nano-30b-a3b"

# We choose this alias to be descriptive for our use case.
MODEL_ALIAS = "nemotron-nano-v3"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider=MODEL_PROVIDER,
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=1.0,
            top_p=1.0,
            max_tokens=2048,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        ),
    )
]

### 🏗️ Initialize the Data Designer Config Builder

- The Data Designer config defines the dataset schema and generation process.

- The config builder provides an intuitive interface for building this configuration.

- The list of model configs is provided to the builder at initialization.



### 🏗️ データデザイナー設定ビルダーの初期化

- データデザイナーの設定では、データセットのスキーマと生成プロセスを定義します。

- 設定ビルダーは、この設定を構築するための直感的なインターフェースを提供します。

- モデル設定のリストは、初期化時にビルダーに渡されます。


In [4]:
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

## 🏥 Prepare a seed dataset

- For this notebook, we'll create a synthetic dataset of patient notes.

- We will _seed_ the generation process with a [symptom-to-diagnosis dataset](https://huggingface.co/datasets/gretelai/symptom_to_diagnosis).

- We already have the dataset downloaded in the [data](../data) directory of this repository.

<br>

> 🌱 **Why use a seed dataset?**
>
> - Seed datasets let you steer the generation process by providing context that is specific to your use case.
>
> - Seed datasets are also an excellent way to inject real-world diversity into your synthetic data.
>
> - During generation, prompt templates can reference any of the seed dataset fields.


## 🏥 シードデータセットの準備

- このノートブックでは、患者の診療記録の合成データセットを作成します。

- 生成プロセスには、[症状から診断までのデータセット](https://huggingface.co/datasets/gretelai/symptom_to_diagnosis)をシードとして使用します。

- このデータセットは、既にこのリポジトリの[data](../data)ディレクトリにダウンロードされています。

<br>

> 🌱 **シードデータセットを使用する理由**
>
> - シードデータセットを使用すると、ユースケースに特化したコンテキストを提供することで、生成プロセスを制御できます。

>
> - シードデータセットは、合成データに現実世界の多様性を取り入れるための優れた方法でもあります。

>
> - 生成中に、プロンプトテンプレートはシードデータセットの任意のフィールドを参照できます。


In [5]:
# Download sample dataset from Github
import urllib.request

url = "https://raw.githubusercontent.com/NVIDIA/GenerativeAIExamples/refs/heads/main/nemo/NeMo-Data-Designer/data/gretelai_symptom_to_diagnosis.csv"
local_filename, _ = urllib.request.urlretrieve(url, "gretelai_symptom_to_diagnosis.csv")

# Seed datasets are passed as reference objects to the config builder.
seed_source = dd.LocalFileSeedSource(path=local_filename)

config_builder.with_seed_dataset(seed_source)

DataDesignerConfigBuilder(
    seed_dataset: local seed
)

## 🎨 Designing our synthetic patient notes dataset

- The prompt template can reference fields from our seed dataset:
  - `{{ diagnosis }}` - the medical diagnosis from the seed data
  - `{{ patient_summary }}` - the symptom description from the seed data

## 🎨 合成患者記録データセットの設計

- プロンプトテンプレートは、シードデータセットのフィールドを参照できます。

- `{{ diagnosis }}` - シードデータからの医学的診断

- `{{ patient_summary }}` - シードデータからの症状の説明


In [6]:
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="patient_sampler",
        sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
        params=dd.PersonFromFakerSamplerParams(),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="doctor_sampler",
        sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
        params=dd.PersonFromFakerSamplerParams(),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="patient_id",
        sampler_type=dd.SamplerType.UUID,
        params=dd.UUIDSamplerParams(
            prefix="PT-",
            short_form=True,
            uppercase=True,
        ),
    )
)

config_builder.add_column(dd.ExpressionColumnConfig(name="first_name", expr="{{ patient_sampler.first_name }}"))

config_builder.add_column(dd.ExpressionColumnConfig(name="last_name", expr="{{ patient_sampler.last_name }}"))

config_builder.add_column(dd.ExpressionColumnConfig(name="dob", expr="{{ patient_sampler.birth_date }}"))

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="symptom_onset_date",
        sampler_type=dd.SamplerType.DATETIME,
        params=dd.DatetimeSamplerParams(start="2024-01-01", end="2024-12-31"),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="date_of_visit",
        sampler_type=dd.SamplerType.TIMEDELTA,
        params=dd.TimeDeltaSamplerParams(dt_min=1, dt_max=30, reference_column_name="symptom_onset_date"),
    )
)

config_builder.add_column(dd.ExpressionColumnConfig(name="physician", expr="Dr. {{ doctor_sampler.last_name }}"))

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="physician_notes",
        prompt="""\
You are a primary-care physician who just had an appointment with {{ first_name }} {{ last_name }},
who has been struggling with symptoms from {{ diagnosis }} since {{ symptom_onset_date }}.
The date of today's visit is {{ date_of_visit }}.

{{ patient_summary }}

Write careful notes about your visit with {{ first_name }},
as Dr. {{ doctor_sampler.first_name }} {{ doctor_sampler.last_name }}.

Format the notes as a busy doctor might.
Respond with only the notes, no other text.
""",
        model_alias=MODEL_ALIAS,
    )
)

data_designer.validate(config_builder)

[11:01:41] [INFO] ✅ Validation passed


### 🔁 Iteration is key – preview the dataset!

1. Use the `preview` method to generate a sample of records quickly.

2. Inspect the results for quality and format issues.

3. Adjust column configurations, prompts, or parameters as needed.

4. Re-run the preview until satisfied.



### 🔁 繰り返し検証が鍵です – データセットをプレビューしましょう！

1. `preview` メソッドを使用して、レコードのサンプルをすばやく生成します。

2. 結果の品質とフォーマットに問題がないか確認します。

3. 必要に応じて、列の設定、プロンプト、またはパラメータを調整します。

4. 満足できるまでプレビューを繰り返し実行します。


In [7]:
preview = data_designer.preview(config_builder, num_records=2)

[11:02:00] [INFO] 📺 Preview generation in progress
[11:02:00] [INFO]   |-- 🔒 Jinja rendering engine: secure
[11:02:00] [INFO] ✅ Validation passed
[11:02:00] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[11:02:00] [INFO] 🩺 Running health checks for models...
[11:02:00] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[11:02:01] [INFO]   |-- ✅ Passed!
[11:02:01] [INFO] ⚡ DATA_DESIGNER_ASYNC_ENGINE is enabled - using async task-queue preview
[11:02:01] [INFO] 📝 llm-text model config for column 'physician_notes'
[11:02:01] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[11:02:01] [INFO]   |-- model alias: 'nemotron-nano-v3'
[11:02:01] [INFO]   |-- model provider: 'nvidia'
[11:02:01] [INFO]   |-- inference parameters:
[11:02:01] [INFO]   |  |-- generation_type=chat-completion
[11:02:01] [INFO]   |  |-- max_parallel_requests=4
[11:02:01] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enabl

In [8]:
# Run this cell multiple times to cycle through the 2 preview records.
preview.display_sample_record()

                                                 Seed Columns                                                 
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name            ┃ Value                                                                                    ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ diagnosis       │ cervical spondylosis                                                                     │
├─────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┤
│ patient_summary │ I've been having a lot of pain in my neck and back. I've also been having trouble with   │
│                 │ my balance and coordination. I've been coughing a lot and my limbs feel weak.            │
└─────────────────┴──────────────────────────────────────────────────────────────────────────────────────────┘
                                                                                                              
                                                                                                              
                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name               ┃ Value                                                                                 ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ patient_sampler    │ {                                                                                     │
│                    │     'uuid': '9c2b6fff-6117-4c53-8224-b5cb96defe23',                                   │
│                    │     'locale': 'en_US',                                                                │
│                    │     'first_name': 'Shelley',                                                          │
│                    │     'last_name': 'Bradshaw',                                                          │
│                    │     'middle_name': None,                                                              │
│                    │     'sex': 'Female',                                                                  │
│                    │     'street_number': '57755',                                                         │
│                    │     'street_name': 'Nicole Lake',                                                     │
│                    │     'city': 'New Tonyatown',                                                          │
│                    │     'state': 'Missouri',                                                              │
│                    │     'postcode': '08866',                                                              │
│                    │     'age': 29,                                                                        │
│                    │     'birth_date': '1996-06-20',                                                       │
│                    │     'country': 'Poland',                                                              │
│                    │     'marital_status': 'never_married',                                                │
│                    │     'education_level': 'doctorate',                                                   │
│                    │     'unit': '',                                                                       │
│                    │     'occupation': 'Phytotherapist',                                                   │
│                    │     'phone_number': '+1-875-294-2881',                                                │
│                    │     'bachelors_field': 'business'                                                     │
│   

In [9]:
# The preview dataset is available as a pandas DataFrame.
preview.dataset

,diagnosis,patient_summary,patient_sampler,doctor_sampler,patient_id,symptom_onset_date,date_of_visit,first_name,dob,last_name,physician,physician_notes
0,cervical spondylosis,I've been having a lot of pain in my neck and ...,{'uuid': '9c2b6fff-6117-4c53-8224-b5cb96defe23...,{'uuid': '5e59a38d-4cd9-4c04-8c41-8fe63fd9bd5a...,PT-46362632,2024-03-09T00:00:00,2024-03-27T00:00:00,Shelley,1996-06-20,Bradshaw,Dr. Clark,**Visited by: Shelley Bradshaw** \n**Date/Tim...
1,impetigo,I have a rash on my face that is getting worse...,{'uuid': '72789541-8fa3-4d05-aa2a-e083137bca53...,{'uuid': '0af00310-3c59-4c44-b6ce-c5d734f1c17c...,PT-DF1F0647,2024-03-02T00:00:00,2024-03-30T00:00:00,Ashley,1912-06-19,Lopez,Dr. Carpenter,**03/30/2024 10:45 AM – Acute Care Visit – Ash...


### 📊 Analyze the generated data

- Data Designer automatically generates a basic statistical analysis of the generated data.

- This analysis is available via the `analysis` property of generation result objects.


### 📊 生成データの分析

- データデザイナーは、生成データの基本的な統計分析を自動的に生成します。

- この分析結果は、生成結果オブジェクトの `analysis` プロパティから利用できます。

In [10]:
# Print the analysis as a table.
preview.analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 2                               │ 10                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                   ┃       data type ┃             number unique values ┃               sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ patient_sampler               │            dict │                       2 (100.0%) │          person_from_faker │
├───────────────────────────────┼─────────────────┼──────────────────────────────────┼────────────────────────────┤
│ doctor_sampler                │            dict │                       2 (100.0%) │          person_from_faker │
├───────────────────────────────┼─────────────────┼──────────────────────────────────┼────────────────────────────┤
│ patient_id                    │          string │                       2 (100.0%) │                       uuid │
├───────────────────────────────┼─────────────────┼──────────────────────────────────┼────────────────────────────┤
│ symptom_onset_date            │          string │                       2 (100.0%) │                   datetime │
├───────────────────────────────┼─────────────────┼──────────────────────────────────┼────────────────────────────┤
│ date_of_visit                 │          string │                       2 (100.0%) │                  timedelta │
└───────────────────────────────┴─────────────────┴──────────────────────────────────┴────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                       ┃               ┃                            ┃     prompt tokens ┃      completion tokens ┃
┃ column name           ┃     data type ┃       number unique values ┃        per record ┃             per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ physician_notes       │        string │                 2 (100.0%) │     135.0 +/- 5.0 │       1070.5 +/- 326.0 │
└───────────────────────┴───────────────┴────────────────────────────┴───────────────────┴────────────────────────┘
                                                                                                                   
                                                          

### 🆙 Scale up!

- Happy with your preview data?

- Use the `create` method to submit larger Data Designer generation jobs.


### 🆙 スケールアップ！

- プレビューデータに満足いただけましたか？

- `create` メソッドを使用して、より大規模な Data Designer 生成ジョブを送信してください。

In [11]:
results = data_designer.create(config_builder, num_records=10, dataset_name="tutorial-3")

[11:03:47] [INFO] 🎨 Creating Data Designer dataset
[11:03:47] [INFO]   |-- 🔒 Jinja rendering engine: secure
[11:03:47] [INFO] ✅ Validation passed
[11:03:47] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[11:03:47] [INFO] 🩺 Running health checks for models...
[11:03:47] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[11:03:47] [INFO]   |-- ✅ Passed!
[11:03:47] [INFO] ⚡ DATA_DESIGNER_ASYNC_ENGINE is enabled - using async task-queue builder
[11:03:47] [INFO] 📝 llm-text model config for column 'physician_notes'
[11:03:47] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[11:03:47] [INFO]   |-- model alias: 'nemotron-nano-v3'
[11:03:47] [INFO]   |-- model provider: 'nvidia'
[11:03:47] [INFO]   |-- inference parameters:
[11:03:47] [INFO]   |  |-- generation_type=chat-completion
[11:03:47] [INFO]   |  |-- max_parallel_requests=4
[11:03:47] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enabl

In [12]:
# Load the generated dataset as a pandas DataFrame.
dataset = results.load_dataset()

dataset.head()

,diagnosis,patient_summary,patient_sampler,doctor_sampler,patient_id,symptom_onset_date,date_of_visit,dob,first_name,last_name,physician,physician_notes
0,cervical spondylosis,I've been having a lot of pain in my neck and ...,"{'age': 100, 'bachelors_field': 'arts_humaniti...","{'age': 58, 'bachelors_field': 'stem', 'birth_...",PT-B0574F7A,2024-06-09T00:00:00,2024-06-30T00:00:00,1926-01-20,Emma,Harrison,Dr. Rose,**Cervical Spondylosis Follow-up – 2024-06-30*...
1,impetigo,I have a rash on my face that is getting worse...,"{'age': 27, 'bachelors_field': 'no_degree', 'b...","{'age': 24, 'bachelors_field': 'no_degree', 'b...",PT-4368B6AD,2024-12-11T00:00:00,2024-12-21T00:00:00,1998-06-15,Cory,Hill,Dr. Mcdonald,**EMR: Progress Note** **Date/Time:** 2024-1...
2,urinary tract infection,I have been urinating blood. I sometimes feel ...,"{'age': 19, 'bachelors_field': 'no_degree', 'b...","{'age': 48, 'bachelors_field': 'arts_humanitie...",PT-5C1170C7,2024-08-15T00:00:00,2024-08-30T00:00:00,2006-12-05,Brooke,Holmes,Dr. Hoffman,"- 8/30/24 08:15am: Brooke Holmes, 28F, present..."
3,arthritis,I have been having trouble with my muscles and...,"{'age': 100, 'bachelors_field': 'stem_related'...","{'age': 52, 'bachelors_field': 'education', 'b...",PT-76205AAB,2024-10-29T00:00:00,2024-11-16T00:00:00,1925-08-10,Kathryn,Carroll,Dr. Graham,"NOTE: Kathryn Carroll, 45 yo F, s/p car accide..."
4,dengue,I have been feeling really sick. My body hurts...,"{'age': 74, 'bachelors_field': 'arts_humanitie...","{'age': 84, 'bachelors_field': 'stem', 'birth_...",PT-0C111A5E,2024-08-03T00:00:00,2024-08-07T00:00:00,1952-01-07,James,Spencer,Dr. Farrell,**2024-08-07 | 08:45** **Patient:** James Sp...


In [13]:
# Load the analysis results into memory.
analysis = results.load_analysis()

analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 10                              │ 10                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                   ┃       data type ┃             number unique values ┃               sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ patient_sampler               │            dict │                      10 (100.0%) │          person_from_faker │
├───────────────────────────────┼─────────────────┼──────────────────────────────────┼────────────────────────────┤
│ doctor_sampler                │            dict │                      10 (100.0%) │          person_from_faker │
├───────────────────────────────┼─────────────────┼──────────────────────────────────┼────────────────────────────┤
│ patient_id                    │          string │                      10 (100.0%) │                       uuid │
├───────────────────────────────┼─────────────────┼──────────────────────────────────┼────────────────────────────┤
│ symptom_onset_date            │          string │                      10 (100.0%) │                   datetime │
├───────────────────────────────┼─────────────────┼──────────────────────────────────┼────────────────────────────┤
│ date_of_visit                 │          string │                      10 (100.0%) │                  timedelta │
└───────────────────────────────┴─────────────────┴──────────────────────────────────┴────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                       ┃               ┃                            ┃     prompt tokens ┃      completion tokens ┃
┃ column name           ┃     data type ┃       number unique values ┃        per record ┃             per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ physician_notes       │        string │                10 (100.0%) │     117.5 +/- 5.7 │        937.5 +/- 396.2 │
└───────────────────────┴───────────────┴────────────────────────────┴───────────────────┴────────────────────────┘
                                                                                                                   
                                                          

## ⏭️ Next Steps

Check out the following notebook to learn more about:

- [Providing images as context](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)

- [Generating images](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)


## ⏭️ 次のステップ

以下のノートブックを参照して、詳細を確認してください。

- [コンテキストとして画像を提供する](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)

- [画像を生成する](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)
